In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
from pathlib import Path
import subprocess

from astropy.io import fits

# Accessing Reduced SpeX data (thru the index.html)
Hi! This notebook will showcase my (Evan) work in trying to access the info on an object after being reduced with pyspextools! 

From quick inspection of some already reduced data, it seems finished data will output an index.html file that holds a lot of an objects info (in the qa directory).

That makes my life a lot easier! So my main goals then will be:
- Access that data with a python function
- Store that data in a table (Final storage / structure can be optimized, just need to be able to store it somewhere)
- Make a front to back function / script to do this for multiple files!

In [ ]:
# Just gonna set this global vartiable so I can grab later!
REDUCTIONS_DIR = Path("reductions")

MOVING_PS_DF = pd.DataFrame()
FIXED_PS_DF = pd.DataFrame()

for redux_dir in REDUCTIONS_DIR.iterdir():
    # EX_DATA = Path(f"{REDUCTIONS_DIR}/20010312-1")
    INDEX_PATH = Path(f"{redux_dir}/qa/index.html")

    with open(INDEX_PATH) as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        if "Source Name" in line:

            if "moving-ps" in line:

                _info_dict = {
                "OBSERVATION_FOLDER" : str(EX_DATA).split("/")[-1],
                "SOURCE NAME" : line.strip("   <li> Source Name:").split(" (moving-ps)")[0],
                "COORDINATES" : lines[i+1].strip("    <li>Coordinate: ").rstrip(" \n")
                }

                MOVING_PS_DF = pd.concat([MOVING_PS_DF, pd.DataFrame([_info_dict])])


            elif "fixed-ps" in line:

                _info_dict = {
                "OBSERVATION_FOLDER" : str(EX_DATA).split("/")[-1],
                "SOURCE NAME" : line.strip("   <li> Source Name:").split(" (fixed-ps)")[0],
                "COORDINATES" : lines[i+1].strip("    <li>Coordinate: ").split(" [<a href=")[0]
                }

                FIXED_PS_DF = pd.concat([FIXED_PS_DF, pd.DataFrame([_info_dict])])

            else:
                print(f"SOMETHING WENT WRONG WITH LINE {i}")

print(MOVING_PS_DF)
FIXED_PS_DF

   OBSERVATION_FOLDER      SOURCE NAME  \
0          20010312-1        1999 NV27   
0          20010312-1        Keet Seel   
0          20010312-1          1994 CO   
0          20010312-1    824 Anastasia   
0          20010312-1          44 Nysa   
..                ...              ...   
0          20010312-1        2001 SG10   
0          20010312-1       719 Albert   
0          20010312-1  1904 Massevitch   
0          20010312-1        1998 SF36   
0          20010312-1     4159 Freeman   

                                         COORDINATES  
0         23:49:47.03 +17:40:36.8 at MJD 52175.28398  
0         00:34:56.64 +13:58:35.1 at MJD 52175.36545  
0          00:54:33.2 -03:30:50.8 at MJD 52175.39984  
0         19:24:44.44 -20:29:49.2 at MJD 52181.20912  
0         19:44:29.05 -21:29:36.9 at MJD 52181.22912  
..                                               ...  
0         00:52:11.45 +00:56:52.6 at MJD 52188.42046  
0         02:43:04.75 +03:46:55.2 at MJD 52188.55069  


,OBSERVATION_FOLDER,SOURCE NAME,COORDINATES
0,20010312-1,66 Maja,03:36:07.39 +21:39:16.7
0,20010312-1,4 Vesta,04:50:32.01 +15:10:59.3
0,20010312-1,673,14:07:35.03 -12:35:37.5
0,20010312-1,1350,21:42:46.14 -12:45:59.1
0,20010312-1,HD214558,22:38:18.02 +45:10:55.5
0,20010312-1,Ennomos,03:31:48.41 +32:13:29.6
0,20010312-1,HD 489,00:09:28.41 +19:06:58.3
0,20010312-1,sao 98291,09:01:03.49 +12:32:36.4
0,20010312-1,446 Aeternitas,22:25:14.6 -25:09:23.3
0,20010312-1,AO128191,23:27:20.97 +04:27:03.3


# Accessing Reductions (thru .fits files)

Post-Week 2 Meeting, I learned that info on the object can also be found in the headers of the spectra's .fits files
_THATS REAL AWKWARD_

That means the index files are really needed? Although I guess it was a nice exercise in a way--knowing I could read thru a file with python via path commands!

In [3]:
hdu = fits.open("/Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/combined/spex-prism_2M-1000+32_2030201001_2009B069_178-197comb.fits")
h = hdu[0].header
hdu.close()
print(h)

SIMPLE  =                    T / conforms to FITS standard                      BITPIX  =                  -64 / array data type                                NAXIS   =                    3 / number of array dimensions                     NAXIS1  =                  563                                                  NAXIS2  =                    4                                                  NAXIS3  =                    1                                                  DIVISOR =                   32 / Normalization value                            ORIGIN  = 'Institute for Astronomy'                                             TELESCOP= 'NASA IRTF'                                                           OBSERVER= 'Your_Name'                                                           PROG_ID =                                                                       OBJECT  = '2M 1000+32'                                                          NDR     =                   32 / Number 

In [ ]:
hdu = fits.open("/Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/reductions/20010621-1/proc/combspec1022-1037.fits")
h = hdu[0].header
hdu.close()
print(h)

SIMPLE  =                    T / conforms to FITS standard                      BITPIX  =                  -64 / array data type                                NAXIS   =                    3 / number of array dimensions                     NAXIS1  =                  563                                                  NAXIS2  =                    4                                                  NAXIS3  =                    1                                                  DIVISOR =                   32 / Normalization value                            ORIGIN  = 'Institute for Astronomy'                                             TELESCOP= 'NASA IRTF'                                                           OBSERVER= 'Bus     '                                                            OBJECT  = '1999 KW4'                                                            BEAM    = 'A       '           / Object(A) or sky(B)                            CYCLES  =                    5 / Number 

In [5]:
hdu = fits.open("/Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/combined/spex-prism_AUR-165_2012201011_2010B106_161-170comb.fits")
h = hdu[0].header
hdu.close()
print(h)

SIMPLE  =                    T / conforms to FITS standard                      BITPIX  =                  -64 / array data type                                NAXIS   =                    3 / number of array dimensions                     NAXIS1  =                  563                                                  NAXIS2  =                    4                                                  NAXIS3  =                    1                                                  DIVISOR =                   32 / Normalization value                            ORIGIN  = 'Institute for Astronomy'                                             TELESCOP= 'NASA IRTF'                                                           OBSERVER= 'Peterson'                                                            PROG_ID =                                                                       OBJECT  = 'AUR-165 '                                                            NDR     =                   32 / Number 

In [6]:
hdu = fits.open("/Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/combined/spex-prism_WISE-0714-1212_20101218_2010B095_53-58.fits")
h = hdu[0].header
hdu.close()
print(h)

SIMPLE  =                    T / conforms to FITS standard                      BITPIX  =                  -64 / array data type                                NAXIS   =                    3 / number of array dimensions                     NAXIS1  =                  563                                                  NAXIS2  =                    4                                                  NAXIS3  =                    1                                                  DIVISOR =                   32 / Normalization value                            ORIGIN  = 'Institute for Astronomy'                                             TELESCOP= 'NASA IRTF'                                                           OBSERVER= 'Your_Name'                                                           PROG_ID =                                                                       OBJECT  = 'WISE 0714-1212'                                                      TABLE_MS=              512.452 / msec to

In [7]:
hdu = fits.open("/Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/combined/spex-sxd_cII-260_2017201002_2010A095_2046-2047comb.fits")
h = hdu[0].header
hdu.close()
print(h)

SIMPLE  =                    T / conforms to FITS standard                      BITPIX  =                  -64 / array data type                                NAXIS   =                    3 / number of array dimensions                     NAXIS1  =                 1014                                                  NAXIS2  =                    4                                                  NAXIS3  =                    6                                                  DIVISOR =                   32 / Normalization value                            ORIGIN  = 'Institute for Astronomy'                                             TELESCOP= 'NASA IRTF'                                                           OBSERVER= 'Your_Name'                                                           PROG_ID =                                                                       OBJECT  = 'cII-260 '                                                            NDR     =                   32 / Number 

In [8]:
hdu = fits.open("/Users/epritchard/Research/CSL_Summer_2026/SpeX_Summer_2026/combined/spex-sxd_P-2010H2_20100422_2010A999_55-64.fits")
h = hdu[0].header
hdu.close()
print(h)

SIMPLE  =                    T / conforms to FITS standard                      BITPIX  =                  -64 / array data type                                NAXIS   =                    3 / number of array dimensions                     NAXIS1  =                 1014                                                  NAXIS2  =                    4                                                  NAXIS3  =                    6                                                  DIVISOR =                   32 / Normalization value                            ORIGIN  = 'Institute for Astronomy'                                             TELESCOP= 'NASA IRTF'                                                           OBSERVER= 'Your_Name'                                                           PROG_ID =                                                                       OBJECT  = 'P/2010H2'                                                            TABLE_MS=              512.452 / msec to

# OKAY lets do more script making! 

Imma run off the assumption that the best indicator of moving versus fixed objects is the units of the y_axis (checking standards sometimes works but other times its iffy)

(ND/s --> reflectance? --> moving source)

(W m-2 um-1 --> energy flux --> fixed source)

In [9]:
# Some setup o7
COMBINED_DIR = Path("combined")

MOVING_PS_DF = pd.DataFrame()
FIXED_PS_DF = pd.DataFrame()

In [15]:
hdu = fits.open(f"{COMBINED_DIR}/spex-sxd_P-2010H2_20100422_2010A999_55-64.fits")
h = hdu[0].header
hdu.close()
h["YUNITS"]

'DN s-1'

In [13]:
help(h)

Help on Header in module astropy.io.fits.header object:

class Header(builtins.object)
 |  Header(cards=[], copy=False)
 |
 |  FITS header class.  This class exposes both a dict-like interface and a
 |  list-like interface to FITS headers.
 |
 |  The header may be indexed by keyword and, like a dict, the associated value
 |  will be returned.  When the header contains cards with duplicate keywords,
 |  only the value of the first card with the given keyword will be returned.
 |  It is also possible to use a 2-tuple as the index in the form (keyword,
 |  n)--this returns the n-th value with that keyword, in the case where there
 |  are duplicate keywords.
 |
 |  For example::
 |
 |      >>> header['NAXIS']
 |      0
 |      >>> header[('FOO', 1)]  # Return the value of the second FOO keyword
 |      'foo'
 |
 |  The header may also be indexed by card number::
 |
 |      >>> header[0]  # Return the value of the first card in the header
 |      'T'
 |
 |  Commentary keywords such as HISTO

In [14]:
h

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -64 / array data type                                
NAXIS   =                    3 / number of array dimensions                     
NAXIS1  =                 1014                                                  
NAXIS2  =                    4                                                  
NAXIS3  =                    6                                                  
DIVISOR =                   32 / Normalization value                            
ORIGIN  = 'Institute for Astronomy'                                             
TELESCOP= 'NASA IRTF'                                                           
OBSERVER= 'Your_Name'                                                           
PROG_ID =                                                                       
OBJECT  = 'P/2010H2'                                                            
TABLE_MS=              512.4

In [34]:
MASTER_DF = pd.DataFrame()

# Helpful Identifiers from .fits:
h["OBSERVER"]
h["OBJECT"]
h["RA"] 
h["DEC"]
# for asteroids:
h["SRT_MJD"]
h["END_MJD"]


# Important info from .fits
h["MODE"]
h["SRT_AM"]
h["END_AM"]
h["SRT_TIME"]
h["END_DATE"]
h["ORDERS"]

# Moving versus fixed:
h["YUNITS"]
h["TC_STDID"]
h["TC_STDST"]


# First lets loop thru all the availible files in the drive so far!
for file in COMBINED_DIR.iterdir():

    _info_dict = {
        "OBSERVER": None,
        "OBJECT" : None,
        "RA" : None,
        "DEC" : None,
        "SRT_MJD"  : None,
        "END_MJD" : None,
        "MODE" : None,
        "SRT_AM" : None,
        "END_AM" : None,
        "SRT_TIME" : None,
        "END_DATE" : None,
        "ORDER" : None,
        "YUNITS" : None,
        "TC_STDID" : None,
        "TC_STDST" : None
        }

    # For each object we will read in their header and immediately close the file to minimize RAM usage
    hdu = fits.open(f"{file}")
    h = hdu[0].header
    hdu.close()

    # To start, we will distinguish between fixed and moving sources then grab some important info about the data reduction / spectra availible.
    # The main indicator I am using for fixed vs moving objects is the units of the observation: either flux density or reflectance
    # Flux density --> star or galaxy --> fixed object
    # Reflectance --> asteroid --> moving object
    # Will also use tc_type (Telluric correction type) as it also gives reflectance or A0V

    for key in _info_dict.keys():

        try:
            _info_dict[key] = h[key]
        except:
            print(f"Something went wrong with {key} in {file}")

    if h["YUNITS"] == "W m-2 um-1":
        _info_dict["OBJECT_TYPE"] = "Fixed"
        # _info_dict["YUNITS"] = h["YUNITS"]

    elif h["YUNITS"] == "DN s-1" or h["YUNITS"] == "reflectance":
        _info_dict["OBJECT_TYPE"] = "Moving"
        # _info_dict["YUNITS"] = h["YUNITS"]

    # print(_info_dict)

    MASTER_DF = pd.concat((MASTER_DF, pd.DataFrame(_info_dict, index=[0])), ignore_index=True)


    
    # if h["YUNITS"] not in units_list:
    #     units_list.append(h["YUNITS"])
    # if h["TC_TYPE"] not in tc_type:
    #     tc_type.append(h["TC_TYPE"])

MASTER_DF

Something went wrong with ORDER in combined/spex-sxd_phi-Per_20090110_2008B083_1-16comb.fits
Something went wrong with ORDER in combined/spex-prism_J0850+10_20061118_2006B072_233-238comb.fits
Something went wrong with ORDER in combined/spex-prism_J1254+4346_20100717_2010A999_1-10comb.fits
Something went wrong with ORDER in combined/spex-sxd_delta-Sco_20090110_2008B083_529-552comb.fits
Something went wrong with ORDER in combined/spex-sxd_V838-Mon_20090110_2008B083_261-276comb.fits
Something went wrong with ORDER in combined/spex-prism_Tmove3-id150431_20100715_2010A999_128-133comb.fits
Something went wrong with ORDER in combined/spex-sxd_phi-Per_20091122_2009B047_209-224comb.fits
Something went wrong with ORDER in combined/spex-prism_0116-1357_20091108_2009B009_83-88comb.fits
Something went wrong with ORDER in combined/spex-sxd_HD283572_2006201001_2009B086_30-66comb.fits
Something went wrong with ORDER in combined/spex-sxd_IRAS-16240-2430-W_20090414_nan_1-4.fits
Something went wrong with

,OBSERVER,OBJECT,RA,DEC,SRT_MJD,END_MJD,MODE,SRT_AM,END_AM,SRT_TIME,END_DATE,ORDER,YUNITS,TC_STDID,TC_STDST,OBJECT_TYPE
0,"E Hesselbach, K. Bjorkman",phi Per,+01:43:38.70,++50:41:22.3,54841.342851,54841.34587,ShortXD,1.585,1.608,08:13:42.334733,2009-01-10,None,W m-2 um-1,HD12365,A0V,Fixed
1,Your_Name,2mass 0850+10: L/T binary,08:50:35.28,+10:57:15.5,54057.612701,54057.624472,LowRes15,1.025,1.016,14:42:17.386584,2006-11-18,None,W m-2 um-1,hd 74721,A0V,Fixed
2,Your_Name,redred-id80435,+12:54:38.19,++43:46:54.2,55394.240318,55394.255071,LowRes15,1.24,1.299,05:46:03.469897,2010-07-17,None,W m-2 um-1,HD 109615,A0V,Fixed
3,"E Hesselbach, K. Bjorkman",delta Sco,+16:00:20.46,-22:37:19.5,54841.637731,54841.638719,ShortXD,2.777,2.744,15:18:19.967282,2009-01-10,None,W m-2 um-1,HD 131951,A0V,Fixed
4,"E Hesselbach, K. Bjorkman",V838 Mon,+07:04:04.47,-03:50:53.9,54841.469131,54841.482366,ShortXD,1.147,1.184,11:15:32.942429,2009-01-10,None,W m-2 um-1,HD 50931,A0V,Fixed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,"Vishnu Reddy, Will Swearson",423 Diotima,+04:27:28.78,++17:18:08.3,55088.535633,55088.593854,LowRes15,1.201,1.031,12:51:18.722121,2009-09-14,None,DN s-1,SAO 93936,G2+V,Moving
535,Looper,TWA 30B,+11:32:17.96,-30:18:23.2,55223.634022,55223.651826,LowRes15,1.808,1.971,15:12:59.466583,2010-01-27,None,DN s-1,HD 98949,A0V,Moving
536,Your_Name,bPic13774,+22:17:16.77,++23:09:26.1,55394.645792,55394.646745,ShortXD,1.23,1.234,15:29:56.464737,2010-07-17,None,W m-2 um-1,HD 212643,A0V,Fixed
537,"E Hesselbach, K. Bjorkman",nu Gem,+06:28:57.19,++20:12:43.1,54841.499811,54841.504701,ShortXD,1.219,1.245,11:59:43.651338,2009-01-10,None,W m-2 um-1,HD 43583,A0V,Fixed


In [28]:
MASTER_DF

""
